# PartLabeler on Google Colab

<a href="https://colab.research.google.com/github/buddi0812/partlabeler/blob/main/PartLabeler_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PartLabeler helps you build an **annotated dataset** (boxes around parts) from a video or a folder of images.
You click on a part, the AI outlines it and follows it through the video, you fix what it gets wrong,
and at the end you export the dataset (YOLO, COCO, CVAT, Pascal VOC or Label Studio) to train your own model.

**How to use this notebook**
- Run the steps from top to bottom: click into a grey code box and press **Shift + Enter** (or the ▶ button).
- Some steps have a small **form** on the right: type your values there before running the step.
- The first time takes about 10 minutes (installing and downloading the AI models). After that, starting is quick.

## Step 1 · Check that you have a GPU

The AI models need a graphics card (GPU) to be fast. Colab lends you one for free:

**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**

Then run the cell below. You should see a table that mentions **Tesla T4**.
If it says that no GPU was found, change the runtime type as above and run it again.

In [ ]:
#@title Step 1: check the GPU
!nvidia-smi || echo "No GPU found. Use Runtime > Change runtime type > T4 GPU, then run this cell again."

## Step 2 · Keep your work on Google Drive (recommended)

Colab forgets everything on its own disk when the session ends. If you connect your Google Drive,
your projects and exported datasets are saved in **My Drive → PartLabeler** and are still there next time.

Colab will open a window asking for permission to access your Drive. Choose your Google account and allow it.
Untick `USE_DRIVE` if you only want to try things out.

Put your videos or image folders on Drive too (for example in `My Drive/PartLabeler/videos`), so you can use them in Step 6.

In [ ]:
#@title Step 2: connect Google Drive (optional)
USE_DRIVE = True  #@param {type:"boolean"}

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    HOME_DIR = "/content/drive/MyDrive/PartLabeler"   # projects persist here
else:
    HOME_DIR = "/content/PartLabeler"                 # temporary: deleted when the session ends

os.makedirs(HOME_DIR, exist_ok=True)
print("Your projects will be saved in", HOME_DIR)

## Step 3 · Download and install PartLabeler

This copies PartLabeler from GitHub and installs the Python packages it needs (2 to 5 minutes).
Colab's own PyTorch (with GPU support) is kept as it is.

- `REPO_URL`: the GitHub address of PartLabeler. Leave it as it is unless you use your own fork.
- `BRANCH`: leave it at `main` unless you were told otherwise.

If Colab then shows **"Restart session"**, click it and continue with Step 4 (no need to run Steps 1 to 3 again).

In [ ]:
#@title Step 3: download and install PartLabeler
REPO_URL = "https://github.com/buddi0812/partlabeler.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/partlabeler"

import os

if not REPO_URL.startswith("https://"):
    raise SystemExit("REPO_URL should be the https address of the PartLabeler repository.")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("PartLabeler is already downloaded; getting the latest version.")
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!bash install.sh --no-models
if globals().get("_exit_code", 0) != 0:
    raise SystemExit("The installation did not finish. Scroll up to the first red ERROR line to see why.")

## Step 4 · Download the AI models

This downloads **SAM 3** (finds and follows parts, about 3.5 GB) and **DINOv3** (suggests similar parts).
Every file is checked against its official checksum. It takes 2 to 4 minutes and has to be repeated
in each new Colab session (Colab's disk is emptied when the session ends).

In [ ]:
#@title Step 4: download the models (SAM 3 and DINOv3)
%cd /content/partlabeler
!python -m engine.models

## Step 5 · Allow interactive widgets

The annotator is an interactive picture that runs inside this notebook.
Colab needs this switch to show such widgets. Run it once per session.

In [ ]:
#@title Step 5: turn on interactive widgets
from google.colab import output
output.enable_custom_widget_manager()
print("Interactive widgets are on.")

## Step 6 · Create or open a project, and annotate

A **project** is one video (or one folder of images) plus your list of part names (classes).

- `USE_SAMPLE`: try PartLabeler with the small sample video that comes with it (a panel of circuit
  boards, from Pexels). Untick it to use your own `SOURCE` and `CLASSES`.
- `PROJECT_NAME`: any short name. If a project with this name exists, it is simply opened again
  (all your boxes are kept), and the other fields are ignored.
- `SOURCE`: the video file (for example `.mp4`) **or** the folder of images, e.g. on your Drive.
- `CLASSES`: the part names, separated by commas (`bolt, nut, washer`),
  or the path of a `classes.txt` (one name per line) or `data.yaml` file.
- `EVERY_N`: for a video, keep every Nth frame (5 = every fifth frame). Smaller = more frames to check.
- `LABELS_DIR` (optional): a folder of existing YOLO labels to bring in and review, for example the
  output of Transfer in Step 8. Only used when the project is created.

When the annotator appears: click on a part to outline it, press **T** to follow it through the video,
fix what drifts, and press **Enter** to confirm a frame. Click on the picture first so it receives your
key presses. Press **?** for every shortcut, or ask **Rivet**, the little robot in the annotator's top bar
(or press **H**), how to do something. Every change is saved immediately.

In [ ]:
#@title Step 6: create or open a project, then annotate
USE_SAMPLE = True  #@param {type:"boolean"}
PROJECT_NAME = "sample_pcb"  #@param {type:"string"}
SOURCE = "/content/drive/MyDrive/PartLabeler/videos/example.mp4"  #@param {type:"string"}
CLASSES = "board, chip"  #@param {type:"string"}
EVERY_N = 5  #@param {type:"integer"}
LABELS_DIR = ""  #@param {type:"string"}

import os, sys

REPO_DIR = "/content/partlabeler"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if "HOME_DIR" not in globals():   # the session was restarted after Step 2
    HOME_DIR = "/content/drive/MyDrive/PartLabeler" if os.path.isdir("/content/drive/MyDrive") else "/content/PartLabeler"

from engine.project import Project, read_classes
from ui.host_widget import Annotator

PROJECT_DIR = os.path.join(HOME_DIR, "projects", PROJECT_NAME.strip())
if os.path.exists(os.path.join(PROJECT_DIR, "project.json")):
    print("Opening the existing project", PROJECT_DIR)
    project = Project(PROJECT_DIR)
else:
    source = os.path.join(REPO_DIR, "samples", "pcb_panel.mp4") if USE_SAMPLE else SOURCE.strip()
    if not os.path.exists(source):
        raise SystemExit(f"SOURCE not found: {source}\nPut the path of a video file or of an image folder into SOURCE.")
    if os.path.isfile(CLASSES.strip()):
        classes = read_classes(CLASSES.strip())
    else:
        classes = [name.strip() for name in CLASSES.split(",") if name.strip()]
    if not classes:
        raise SystemExit("Please give at least one class name in CLASSES.")
    is_video = os.path.isfile(source)
    print(f"Creating {PROJECT_DIR}\n  from the {'video' if is_video else 'image folder'} {source}\n  classes: {classes}")
    project = Project.create(PROJECT_DIR, classes,
                             video=source if is_video else None,
                             images=None if is_video else source,
                             every=int(EVERY_N),
                             progress=lambda i, n, text: print(f"\r  {text} {i}/{n}", end=""))
    print()
    if LABELS_DIR.strip():
        print("Imported labels:", project.import_yolo(LABELS_DIR.strip()))

annotator = Annotator(project)
annotator

## Step 7 · Export the dataset

Writes your boxes as a dataset into the project's `exports` folder (on your Drive if you connected it).

- `FORMAT`: `yolo` (Ultralytics YOLO, RF-DETR), `coco` (COCO json), `cvat`, `voc` (Pascal VOC) or `labelstudio`.
- `REVIEWED_ONLY`: only frames you confirmed with **Enter** (recommended: those are the ones you checked).
- `ZIP_AND_DOWNLOAD`: also pack the export into a .zip and download it to your computer.

In [ ]:
#@title Step 7: export the dataset
FORMAT = "yolo"  #@param ["yolo", "coco", "cvat", "voc", "labelstudio"]
REVIEWED_ONLY = True  #@param {type:"boolean"}
ZIP_AND_DOWNLOAD = False  #@param {type:"boolean"}

import os, shutil, time

if "PROJECT_DIR" not in globals():
    raise SystemExit("Run Step 6 first (it opens the project).")

from engine.project import Project

out_dir = os.path.join(PROJECT_DIR, "exports", f"{FORMAT}_{time.strftime('%Y%m%d_%H%M%S')}")
result = Project(PROJECT_DIR).export(FORMAT, out_dir, reviewed_only=REVIEWED_ONLY)
print(f"Exported {result['images']} images and {result['boxes']} boxes to\n  {out_dir}")

if ZIP_AND_DOWNLOAD:
    from google.colab import files
    archive = shutil.make_archive(f"/content/{os.path.basename(out_dir)}", "zip", out_dir)
    files.download(archive)

## Step 8 · Teach & Transfer (optional)

Once one video is fully labeled, PartLabeler can **learn** from it and **label similar videos automatically**:

1. **Teach** trains a small detector (RF-DETR) on a dataset you exported in Step 7 (YOLO format).
   On the free T4 this takes roughly half an hour to a few hours, depending on the number of images and epochs.
   The result is saved in `PartLabeler/runs/RUN_NAME`, so it survives a disconnect if you use Drive.
2. **Transfer** uses that detector to label new videos or image folders. The results go to
   `PartLabeler/transfer/OUT_NAME`, one folder per video, with a `preview.mp4` to watch and a
   `summary.json` that lists the frames to check (for example where a part's label flips along the video).
3. **Check the result**: for each video, Transfer also prepares a project called `<video name>_check`.
   Open it in Step 6 by typing that name into `PROJECT_NAME`, then review and fix the boxes as usual.

No time to train? **Step 8c, quick transfer** labels new videos straight from a labeled dataset, without
training: a rough first pass (it found about 70% of the parts in our tests) that you then check and fix.

Training never starts by itself: tick `RUN_TEACH` (or `RUN_TRANSFER`, `RUN_QUICK`) and run the cell.

- `DATASET`: an exported dataset folder. Leave it empty to use the newest YOLO export of the project from Step 6.
- `PARENT` (optional): the object that holds the parts (for example `machine` or `engine block`).
  PartLabeler then finds that object first and looks for the parts inside it.
- `SIZE`: model size. `small` is a good start; `nano` is faster, `medium` more accurate but slower.
- `EPOCHS`: how many times training goes through the dataset (30 is a good start).

In [ ]:
#@title Step 8a: Teach - train a detector on your exported dataset
RUN_TEACH = False  #@param {type:"boolean"}
DATASET = ""  #@param {type:"string"}
RUN_NAME = "run1"  #@param {type:"string"}
PARENT = ""  #@param {type:"string"}
SIZE = "small"  #@param ["nano", "small", "medium"]
EPOCHS = 30  #@param {type:"integer"}

import glob, os, shlex

os.chdir("/content/partlabeler")
if "HOME_DIR" not in globals():
    HOME_DIR = "/content/drive/MyDrive/PartLabeler" if os.path.isdir("/content/drive/MyDrive") else "/content/PartLabeler"

dataset = DATASET.strip()
if not dataset and "PROJECT_DIR" in globals():
    exports = sorted(glob.glob(os.path.join(PROJECT_DIR, "exports", "yolo_*")))
    dataset = exports[-1] if exports else ""
RUN_DIR = os.path.join(HOME_DIR, "runs", RUN_NAME.strip())

if not RUN_TEACH:
    print("Training is off. Tick RUN_TEACH and run this cell again to start it.")
elif not dataset or not os.path.isdir(dataset):
    raise SystemExit("No dataset found: export one in Step 7 (YOLO format), or put its folder into DATASET.")
else:
    args = ["teach", dataset, "--run", RUN_DIR, "--size", SIZE, "--epochs", str(int(EPOCHS))]
    if PARENT.strip():
        args += ["--parent", PARENT.strip()]
    cmd = " ".join(shlex.quote(a) for a in args)
    print("Training on", dataset, "\nResults go to", RUN_DIR)
    !python -m engine.cli {cmd}

In [ ]:
#@title Step 8b: Transfer - label new videos with the trained detector
RUN_TRANSFER = False  #@param {type:"boolean"}
RUN_NAME = "run1"  #@param {type:"string"}
SOURCES = "/content/drive/MyDrive/PartLabeler/videos/new_video.mp4"  #@param {type:"string"}
OUT_NAME = "auto_labels"  #@param {type:"string"}
#@markdown `SOURCES`: one or more videos or image folders, separated by commas.

import os, shlex

os.chdir("/content/partlabeler")
if "HOME_DIR" not in globals():
    HOME_DIR = "/content/drive/MyDrive/PartLabeler" if os.path.isdir("/content/drive/MyDrive") else "/content/PartLabeler"

RUN_DIR = os.path.join(HOME_DIR, "runs", RUN_NAME.strip())
OUT_DIR = os.path.join(HOME_DIR, "transfer", OUT_NAME.strip())
sources = [s.strip() for s in SOURCES.split(",") if s.strip()]

if not RUN_TRANSFER:
    print("Transfer is off. Tick RUN_TRANSFER and run this cell again to start it.")
elif not os.path.isdir(RUN_DIR):
    raise SystemExit(f"No trained detector in {RUN_DIR}: run Step 8a first, or check RUN_NAME.")
elif not sources or not all(os.path.exists(s) for s in sources):
    raise SystemExit("Some SOURCES were not found: " + ", ".join(s for s in sources if not os.path.exists(s)))
else:
    cmd = " ".join(shlex.quote(a) for a in ["transfer", RUN_DIR, *sources, "--out", OUT_DIR])
    !python -m engine.cli {cmd}
    # One folder per source in OUT_DIR (images/, labels/, preview.mp4). Make a project to check each video.
    for src in sources:
        name = os.path.splitext(os.path.basename(src.rstrip("/")))[0]
        labels = os.path.join(OUT_DIR, name, "labels")
        if not os.path.isdir(labels):
            continue
        if os.path.isfile(src):
            check_dir = os.path.join(HOME_DIR, "projects", name + "_check")
            if not os.path.exists(os.path.join(check_dir, "project.json")):
                cmd = " ".join(shlex.quote(a) for a in ["review", check_dir, "--video", src, "--labels", labels,
                                                         "--classes", os.path.join(OUT_DIR, name, "classes.txt")])
                !python -m engine.cli {cmd}
            print(f"To check {name}: in Step 6 untick USE_SAMPLE, set PROJECT_NAME = {name}_check and run it.")
        else:
            print(f"To check {name}: in Step 6 use a new PROJECT_NAME, SOURCE = {src}, LABELS_DIR = {labels}.")

In [ ]:
#@title Step 8c: Quick transfer - label new videos without training (rough preview)
RUN_QUICK = False  #@param {type:"boolean"}
DATASET = ""  #@param {type:"string"}
SOURCES = "/content/drive/MyDrive/PartLabeler/videos/new_video.mp4"  #@param {type:"string"}
RUN_NAME = "quick1"  #@param {type:"string"}
PARENT = ""  #@param {type:"string"}
#@markdown `DATASET`: a labeled YOLO dataset (empty: the newest YOLO export of the Step 6 project). No training:
#@markdown about 30 of its labeled images are matched in each new frame, 1 to 2 seconds a frame on the T4.

import glob, os, shlex

os.chdir("/content/partlabeler")
if "HOME_DIR" not in globals():
    HOME_DIR = "/content/drive/MyDrive/PartLabeler" if os.path.isdir("/content/drive/MyDrive") else "/content/PartLabeler"

dataset = DATASET.strip()
if not dataset and "PROJECT_DIR" in globals():
    exports = sorted(glob.glob(os.path.join(PROJECT_DIR, "exports", "yolo_*")))
    dataset = exports[-1] if exports else ""
RUN_DIR = os.path.join(HOME_DIR, "runs", RUN_NAME.strip())
sources = [s.strip() for s in SOURCES.split(",") if s.strip()]

if not RUN_QUICK:
    print("Quick transfer is off. Tick RUN_QUICK and run this cell again to start it.")
elif not dataset or not os.path.isdir(dataset):
    raise SystemExit("No dataset found: export one in Step 7 (YOLO format), or put its folder into DATASET.")
elif not sources or not all(os.path.exists(s) for s in sources):
    raise SystemExit("Some SOURCES were not found: " + ", ".join(s for s in sources if not os.path.exists(s)))
else:
    args = ["quick", dataset, *sources, "--run", RUN_DIR]
    if PARENT.strip():
        args += ["--parent", PARENT.strip()]
    cmd = " ".join(shlex.quote(a) for a in args)
    !python -m engine.cli {cmd}
    for src in sources:
        name = os.path.splitext(os.path.basename(src.rstrip("/")))[0]
        labels = os.path.join(RUN_DIR, "labels", name if os.path.isfile(src) else os.path.basename(src.rstrip("/")), "labels")
        if not os.path.isdir(labels):
            continue
        if os.path.isfile(src):
            check_dir = os.path.join(HOME_DIR, "projects", name + "_quick_check")
            if not os.path.exists(os.path.join(check_dir, "project.json")):
                cmd = " ".join(shlex.quote(a) for a in ["review", check_dir, "--video", src, "--labels", labels,
                                                         "--classes", os.path.join(os.path.dirname(labels), "classes.txt")])
                !python -m engine.cli {cmd}
            print(f"To check {name}: in Step 6 untick USE_SAMPLE, set PROJECT_NAME = {name}_quick_check and run it.")
        else:
            print(f"To check {name}: in Step 6 use a new PROJECT_NAME, SOURCE = {src}, LABELS_DIR = {labels}.")

## Tips

- **Everything runs inside this notebook.** Colab's free tier does not allow using a separate web page
  (such as a Gradio link or a local server) as the main way of working, so on Colab the annotator is
  shown right here, under Step 6. Do not start `partlabeler app` on Colab; that is for your own PC.
- **Save projects on Drive** (Step 2). Colab empties its disk when the session ends, and free sessions
  stop after a while of inactivity (and after at most about 12 hours). Your boxes are saved on every
  change, so nothing you confirmed is lost; just run Steps 2 to 6 again next time.
- **Rivet, the helper robot** (top bar of the annotator, or **H**) answers questions such as "how do I export?".
  For full answers, add a free Gemini API key (from aistudio.google.com/apikey) as a Colab secret: click the
  key icon in the left sidebar, add `GEMINI_API_KEY`, and allow this notebook to use it. Without a key Rivet
  answers from the built-in guide. Only your typed question goes to Google, never your images or labels.
- **The T4 uses fp16 automatically.** PartLabeler picks the number format for the GPU by itself:
  bfloat16 on RTX 30xx and newer, float16 on the T4, float32 on a CPU. You don't need to set anything.
- **Widget not showing?** Run Step 5 again, then Step 6. After *Runtime → Restart session*, run Steps 5 and 6
  (and Step 2 if Drive asks to reconnect).
- **Drive feels slow?** Creating a project from a long video writes many frame images. If that is slow on
  Drive, untick `USE_DRIVE` in Step 2, work on Colab's disk, and use `ZIP_AND_DOWNLOAD` in Step 7 to keep the export.
- **On your own PC** (Windows with an NVIDIA GPU, or Linux), install with `install_windows.bat` or
  `bash install.sh`, then start `run_windows.bat` or `./run.sh`: the same annotator opens in your browser.